# Loss-induced concentration of photonic quantum kernels

Reproducibility notebook. It calls the project library in `src/` (the executable ground truth) and regenerates every reported number and figure. Author: Hitesh Kumar Singh.

Sections: (1) conventions & validation against qutip; (2) exact Gaussian concentration law; (3) loss dependence of the rate; (4) bandwidth reversibility; (5) GBS feature kernel; (6) figures.

In [1]:
import sys, os, pathlib
# locate repo root (dir containing src/gaussian.py) and work from there
_p = pathlib.Path.cwd()
for _c in [_p, *_p.parents]:
    if (_c/'src'/'gaussian.py').exists():
        os.chdir(_c); sys.path.insert(0, str(_c)); break
import numpy as np
from src.gaussian import single_mode_kernel, effective_bandwidth, loss_covariance_diag
from src.concentration import exact_mean_var, exact_rate_per_mode, fit_decay_rate
np.set_printoptions(precision=6, suppress=True)
print('library imported')

library imported


## 1. Conventions and validation
hbar=2 (vacuum covariance = I). The encoding shifts only the displacement, so both states share one covariance and the squared-fidelity kernel is exactly `k = exp(-gamma dx^2)` per mode. We validate against qutip Fock-space fidelity including a beamsplitter loss channel.

In [2]:
import qutip as qt
N=30
def k_qutip(rho1,rho2):
    return qt.fidelity(qt.ket2dm(rho1) if rho1.type=='ket' else rho1,
                       qt.ket2dm(rho2) if rho2.type=='ket' else rho2)**2
def lossy(psi, eta):
    th=np.arccos(np.sqrt(eta))
    a1=qt.tensor(qt.destroy(N),qt.qeye(N)); a2=qt.tensor(qt.qeye(N),qt.destroy(N))
    BS=(th*(a1.dag()*a2 - a1*a2.dag())).expm()
    out=BS*qt.tensor(psi, qt.basis(N,0)); return (out*out.dag()).ptrace(0)
v0=qt.basis(N,0); errs=[]
for r,eta,x,xp in [(0.0,1.0,0.3,-0.2),(0.6,0.5,0.8,0.2),(0.6,0.3,0.4,-0.3),(1.0,1.0,0.0,1.0)]:
    s=qt.squeeze(N,r)
    q1=qt.displace(N,x)*s*v0; q2=qt.displace(N,xp)*s*v0
    if eta<1.0: r1,r2=lossy(q1,eta),lossy(q2,eta)
    else: r1,r2=q1,q2
    k=single_mode_kernel(x-xp,r,eta); kq=k_qutip(r1,r2); errs.append(abs(k-kq))
    print(f'r={r} eta={eta} x={x:+.1f} x\'={xp:+.1f}: k={k:.6f} qutip={kq:.6f} err={abs(k-kq):.2e}')
print('max |error| =', max(errs))
assert max(errs) < 1e-3

r=0.0 eta=1.0 x=+0.3 x'=-0.2: k=0.778801 qutip=0.778801 err=2.90e-10


r=0.6 eta=0.5 x=+0.8 x'=+0.2: k=0.758306 qutip=0.758306 err=1.15e-08


r=0.6 eta=0.3 x=+0.4 x'=-0.3: k=0.830281 qutip=0.830281 err=8.36e-09
r=1.0 eta=1.0 x=+0.0 x'=+1.0: k=0.000618 qutip=0.000614 err=4.37e-06
max |error| = 4.36992769795128e-06


## 2. Exact Gaussian concentration law
For iid Gaussian data, `Var[k] = (1+8 g sigma^2)^(-M/2) - (1+4 g sigma^2)^(-M)`, exponential in M with per-mode rate `c = (1/2) ln(1+8 g sigma^2)`. We confirm the exponential fit quality.

In [3]:
sigma=1.0; disp=1.0; M_values=[1,2,4,8,16,32,50]
for r,eta in [(0.7,1.0),(0.7,0.5),(1.5,1.0)]:
    g=effective_bandwidth(r,eta,disp)
    V=[exact_mean_var(g,M,sigma)[1] for M in M_values]
    slope,inter,r2=fit_decay_rate(M_values,V)
    print(f'r={r} eta={eta}: gamma={g:.3f} rate/mode={exact_rate_per_mode(g,sigma):.4f} '
          f'fit_slope={-slope:.4f} R2={r2:.4f}')

r=0.7 eta=1.0: gamma=4.055 rate/mode=1.7549 fit_slope=1.7510 R2=1.0000
r=0.7 eta=0.5: gamma=0.802 rate/mode=1.0019 fit_slope=0.9892 R2=0.9997
r=1.5 eta=1.0: gamma=20.086 rate/mode=2.5428 fit_slope=2.5414 R2=1.0000


## 3. Loss dependence of the concentration rate
The effective bandwidth gamma increases with transmissivity eta (dgamma/deta = disp^2/v_x^2 > 0), so the per-mode rate increases with eta: loss lowers the rate.

In [4]:
for r in [0.3,0.7,1.0]:
    line=[]
    for eta in [0.1,0.25,0.5,0.75,1.0]:
        g=effective_bandwidth(r,eta,1.0); line.append((eta,exact_rate_per_mode(g,1.0)))
    print(f'r={r}: '+'  '.join(f'eta={e:.2f}->c={c:.3f}' for e,c in line))

r=0.3: eta=0.10->c=0.304  eta=0.25->c=0.590  eta=0.50->c=0.909  eta=0.75->c=1.155  eta=1.00->c=1.373
r=0.7: eta=0.10->c=0.312  eta=0.25->c=0.621  eta=0.50->c=1.002  eta=0.75->c=1.347  eta=1.00->c=1.755
r=1.0: eta=0.10->c=0.314  eta=0.25->c=0.634  eta=0.50->c=1.043  eta=0.75->c=1.447  eta=1.00->c=2.048


## 4. Bandwidth reversibility
Because loss only rescales gamma, increasing the displacement restores the lossless gamma exactly.

In [5]:
r=0.7; g_target=effective_bandwidth(r,1.0,1.0)
for eta in [1.0,0.7,0.5,0.3,0.1]:
    v_x,_=loss_covariance_diag(r,eta); disp_needed=np.sqrt(g_target*v_x/eta)
    g_ach=effective_bandwidth(r,eta,disp_needed)
    print(f'eta={eta}: disp_needed={disp_needed:.4f} -> gamma={g_ach:.4f} (target {g_target:.4f})')

eta=1.0: disp_needed=1.0000 -> gamma=4.0552 (target 4.0552)
eta=0.7: disp_needed=1.6547 -> gamma=4.0552 (target 4.0552)
eta=0.5: disp_needed=2.2484 -> gamma=4.0552 (target 4.0552)
eta=0.3: disp_needed=3.2345 -> gamma=4.0552 (target 4.0552)
eta=0.1: disp_needed=6.1235 -> gamma=4.0552 (target 4.0552)


## 5. GBS feature kernel (small-scale numerical exploration)
Photon-number-probability features via thewalrus, validated against qutip; cosine-similarity kernel. Reported for small M as a finite-size observation.

In [6]:
from src.gbs import gbs_mean_var
rng=np.random.default_rng(20260615+99)
for r in [0.6]:
    for eta in [1.0,0.7,0.4]:
        for M,cut in [(1,10),(2,7),(3,5)]:
            mean,var,(lo,hi)=gbs_mean_var(r,M,eta,cut,0.8,80,2026,rng,1.0,n_boot=80)
            g=effective_bandwidth(r,eta,1.0); _,vg=exact_mean_var(g,M,0.8)
            print(f'r={r} eta={eta} M={M}: GBS Var={var:.3e} [{lo:.2e},{hi:.2e}]  Gaussian Var={vg:.3e}')

r=0.6 eta=1.0 M=1: GBS Var=9.921e-02 [7.87e-02,1.14e-01]  Gaussian Var=1.304e-01


r=0.6 eta=1.0 M=2: GBS Var=8.971e-02 [7.22e-02,1.09e-01]  Gaussian Var=4.448e-02


r=0.6 eta=1.0 M=3: GBS Var=8.665e-02 [7.01e-02,1.01e-01]  Gaussian Var=1.193e-02


r=0.6 eta=0.7 M=1: GBS Var=5.646e-02 [3.45e-02,7.46e-02]  Gaussian Var=1.314e-01


r=0.6 eta=0.7 M=2: GBS Var=7.162e-02 [5.30e-02,8.46e-02]  Gaussian Var=7.554e-02


r=0.6 eta=0.7 M=3: GBS Var=6.170e-02 [4.57e-02,7.55e-02]  Gaussian Var=3.315e-02


r=0.6 eta=0.4 M=1: GBS Var=2.480e-02 [1.47e-02,3.41e-02]  Gaussian Var=9.714e-02


r=0.6 eta=0.4 M=2: GBS Var=4.082e-02 [2.66e-02,6.06e-02]  Gaussian Var=8.967e-02


r=0.6 eta=0.4 M=3: GBS Var=2.906e-02 [1.84e-02,3.96e-02]  Gaussian Var=6.231e-02


## 6. Concentration vs classical-simulability boundary (non-Gaussian sector)
Encoding |chi(r)> = S(r)|1> (squeezed single photon), displaced by data, M-mode product, loss eta. We compute the Wigner negativity (simulability resource: N>0 => classically hard) and the per-mode concentration rate, showing they diverge: concentration persists where the kernel is classically simulable.

In [7]:
from src.nongaussian import lossy_sigma, wigner_negativity, single_mode_moments, concentration_rate
print('  r    eta    negativity N   hard?   conc.rate c')
for r in [0.5]:
    for eta in [0.4, 0.6, 1.0]:
        sig=lossy_sigma(r,eta); N=wigner_negativity(sig)
        Ek,Ek2=single_mode_moments(sig,eta,0.8,n_grid=161); c=concentration_rate(Ek2)
        print(f'  {r}   {eta}     {N:.4f}        {int(N>1e-3)}      {c:.4f}')
print('-> eta=0.4 is classically simulable (N~0) yet still concentrates (c>0): boundaries diverge.')

  r    eta    negativity N   hard?   conc.rate c


  0.5   0.4     0.0000        0      0.5659


  0.5   0.6     0.0112        1      0.7783


  0.5   1.0     0.2133        1      1.8693
-> eta=0.4 is classically simulable (N~0) yet still concentrates (c>0): boundaries diverge.


## 7. Regenerate the figure set
Calls the same figure script and style layer used for the manuscript figures.

In [8]:
import subprocess
print(subprocess.run([sys.executable,'src/make_figures.py'],capture_output=True,text=True).stdout)
print(subprocess.run([sys.executable,'tools/figure_qc.py','--project','.'],capture_output=True,text=True).stdout.splitlines()[-1])

saved figures/fig1_concentration_vs_M.pdf and figures/fig1_concentration_vs_M.png (9.0 cm, single column) [OK]
fig1 done
saved figures/fig2_rate_vs_loss.pdf and figures/fig2_rate_vs_loss.png (9.0 cm, single column) [OK]
fig2 done
saved figures/fig3_bandwidth_compensation.pdf and figures/fig3_bandwidth_compensation.png (9.0 cm, single column) [OK]
fig3 done
saved figures/fig4_gbs_vs_gaussian.pdf and figures/fig4_gbs_vs_gaussian.png (18.0 cm, double column) [OK]
fig4 done
saved figures/fig5_concentration_vs_simulability.pdf and figures/fig5_concentration_vs_simulability.png (18.0 cm, double column) [OK]
fig5 done
saved figures/fig6_nongaussian_concentration.pdf and figures/fig6_nongaussian_concentration.png (9.0 cm, single column) [OK]
fig6 done

FIGURE QC: PASS
